# Imports

In [263]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tqdm
import os
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_squared_log_error

# Loading Data

In [60]:
train = pd.read_csv("data/train_final.csv")
test = pd.read_csv("data/test_final.csv")

In [61]:
train_id = train["id"]
test_id = test["id"]

## Preliminary Analysis

In [62]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 17 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
 6   family_new   object 
 7   day          int64  
 8   week         int64  
 9   month        int64  
 10  year         int64  
 11  isweekend    bool   
 12  wage_day     bool   
 13  cluster_new  object 
 14  type         object 
 15  oil_price    float64
 16  holiday      bool   
dtypes: bool(3), float64(2), int64(7), object(5)
memory usage: 329.1+ MB


Let's drop some unnecessary columns.

In [63]:
train.drop(["store_nbr", "family"], axis=1, inplace=True)
test.drop(["store_nbr", "family"], axis=1, inplace=True)

# Some Thoughts on How to Make Models

## How to Train?

We need to create model which can predict the sale for individual family for each cluster. In fact, in an ideal situation, we should use a model which can predict the sale for each family and for each store. However, in this way, there are going to be a huge number of models which is not feasible.

This is what we'll do:
1. We'll create seperate datasets for each family and cluster.
2. Based on these datasets, we'll create a model for each family and cluster.
3. We'll then make predictions for each family and cluster and merge them together to get the final predictions.

> We don't need to create separate models (by this we mean the architecture of the model) for each family and cluster. We can use the same model for each. Of course, the weights of the models will be different.

## How to Compare?

So, how are we gonna compare which model is the best? For now, we'll just be using average of the predictions.

# Creating Datasets

In [64]:
train["family_new"].nunique(), test["family_new"].nunique()

(8, 8)

In [65]:
train["cluster_new"].nunique(), test["cluster_new"].nunique()

(11, 11)

So, there are 8 different families and 11 different clusters (even after we reduced the number of clusters and families). This means that we'll need to create 8*11 = 88 different datasets and then we have to train model for each of them. Let's get started.

In [66]:
clusters = train["cluster_new"].unique()
families = train["family_new"].unique()

In [67]:
train_shapes = {}
test_shapes = {}
for c in tqdm.tqdm(clusters, desc="Saving Datasets..."):
    for f in families:
        #Getting the train data for the given cluster and family
        train_temp = train[(train["cluster_new"] == c) & (train["family_new"] == f)] 
        test_temp = test.loc[(test["cluster_new"] == c) & (test["family_new"] == f)]

        # Dropping the cluster and family columns
        train_temp = train_temp.drop(["cluster_new", "family_new"], axis=1)
        test_temp = test_temp.drop(["cluster_new", "family_new"], axis=1)

        # Getting the shapes
        train_shapes[f"{c}_{f}"] = train_temp.shape
        test_shapes[f"{c}_{f}"] = test_temp.shape
        
        # Setting the names of the file
        train_file_name = "data/train/" + str(c) + "_" + str(f) + ".csv"
        test_file_name = "data/test/" + str(c) + "_" + str(f) + ".csv"

        # saving the dataframes
        train_temp.to_csv(train_file_name, index=False)
        test_temp.to_csv(test_file_name, index=False)

Saving Datasets...: 100%|██████████| 11/11 [01:00<00:00,  5.49s/it]


Great! Let's save the information about the shape of dataframes into a different dataframe.

In [68]:
train_shapes_df = pd.DataFrame.from_dict(train_shapes).T.reset_index()
train_shapes_df.columns = ["file_name", "length", "width"]

test_shapes_df = pd.DataFrame.from_dict(test_shapes).T.reset_index()
test_shapes_df.columns = ["file_name", "length", "width"]
train_shapes_df

,file_name,length,width
0,E_MISC,40416,13
1,E_LADIES,26944,13
2,E_BEVERAGE,20208,13
3,E_STATIONARY,20208,13
4,E_FOOD,53888,13
...,...,...,...
83,F_STATIONARY,20208,13
84,F_FOOD,53888,13
85,F_HOUSEHOLD,33680,13
86,F_GROCERY,13472,13


In [69]:
# train_shapes_df.to_csv("data/train_shapes.csv", index=False)
# test_shapes_df.to_csv("data/test_shapes.csv", index=False)
train_shapes_df = pd.read_csv("data/train_shapes.csv")
test_shapes_df = pd.read_csv("data/test_shapes.csv")

# Making the Datasets ready for training

## The Window Size

Let's load some of the datasets from the datasets we created.

In [70]:
sample_df_1 = pd.read_csv(r"data\train\A_FOOD.csv")
sample_df_1.groupby("date").count()["sales"]

date
2013-01-01    56
2013-01-02    56
2013-01-03    56
2013-01-04    56
2013-01-05    56
              ..
2017-08-11    56
2017-08-12    56
2017-08-13    56
2017-08-14    56
2017-08-15    56
Name: sales, Length: 1684, dtype: int64

In [71]:
sample_df_2 = pd.read_csv(r"data\train\B_BEVERAGE.csv")
sample_df_2.groupby("date").count()["sales"]

date
2013-01-01    18
2013-01-02    18
2013-01-03    18
2013-01-04    18
2013-01-05    18
              ..
2017-08-11    18
2017-08-12    18
2017-08-13    18
2017-08-14    18
2017-08-15    18
Name: sales, Length: 1684, dtype: int64

We see that there are different number of entries for one day for different datasets. This is because we reduced the number of families and clusters. Let's see how many entries there are for each day for each training and test dataset. We'll be using this information for the window size. 

In [72]:
train_datasets = os.listdir("data/train/")
test_datasets = os.listdir("data/test/")

In [73]:
window_size = {}
for f in train_datasets:
    df = pd.read_csv("data/train/" + f)
    w = df.groupby("date").count()["sales"][0]
    window_size[f] = w
    df2 = pd.read_csv("data/test/" + f)
    w2 = df2.groupby("date").count()["onpromotion"][0]
    if w!=w2:
        print("WindowsError")
        print(f)

In [74]:
window_size_df = pd.DataFrame.from_dict(window_size, orient="index").reset_index()
window_size_df.columns = ["file_name", "window_size"]
window_size_df.to_csv("data/window_size.csv", index=False)

In [75]:
window_size_df = pd.read_csv("data/window_size.csv")
window_size_df.head()

,file_name,window_size
0,A_BEVERAGE.csv,21
1,A_ELECTRONICS.csv,14
2,A_FOOD.csv,56
3,A_GROCERY.csv,14
4,A_HOUSEHOLD.csv,35


We created the datasets based on family and cluster. Now, we need to make them ready for training. We will use a window size such that we get 2-5 days of data to predict, depending on the window size.

## Creating One Dataset Ready to Train

### Deciding the Window Size

Now, we have all the information needed to make the windowed datasets. Let's start.

In [76]:
file = "A_FOOD.csv"
df = pd.read_csv("data/train/" + file)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94304 entries, 0 to 94303
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           94304 non-null  int64  
 1   date         94304 non-null  object 
 2   sales        94304 non-null  float64
 3   onpromotion  94304 non-null  int64  
 4   day          94304 non-null  int64  
 5   week         94304 non-null  int64  
 6   month        94304 non-null  int64  
 7   year         94304 non-null  int64  
 8   isweekend    94304 non-null  bool   
 9   wage_day     94304 non-null  bool   
 10  type         94304 non-null  object 
 11  oil_price    94304 non-null  float64
 12  holiday      94304 non-null  bool   
dtypes: bool(3), float64(2), int64(6), object(2)
memory usage: 7.5+ MB


Let's create a function to get the window size. To get this, we'll use following considerations:
1. The minimum day the window size should cover is 1.
2. The maximum value of the window should be 60 (can be greate if (1) is not satisfied).
3. The maximum number of days in the window should not be greater than 7.

In [77]:
def get_window_size(window_size):
    max_days = 60//window_size
    if max_days>7:
        return 7*window_size
    elif max_days<1:
        return window_size
    else:
        return max_days*window_size

In [78]:
WINDWOW = window_size[file]
WINDOW = get_window_size(WINDWOW)
WINDOW

56

### Using `pandas`' `shift` Method

In [79]:
for i in range(WINDOW):
    df[f"sales_{i+1}"] = df["sales"].shift(i+1)

In [80]:
df = df.dropna()

In [81]:
df

,id,date,sales,onpromotion,day,week,month,year,isweekend,wage_day,...,sales_47,sales_48,sales_49,sales_50,sales_51,sales_52,sales_53,sales_54,sales_55,sales_56
56,2018,2013-01-02,227.000,0,2,1,1,2013,False,False,...,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000
57,2022,2013-01-02,295.000,0,2,1,1,2013,False,False,...,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000
58,2023,2013-01-02,155.000,0,2,1,1,2013,False,False,...,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000
59,2024,2013-01-02,29.000,0,2,1,1,2013,False,False,...,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000
60,2037,2013-01-02,12.584,0,2,1,1,2013,False,False,...,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94299,3000734,2017-08-15,32.000,0,15,33,8,2017,False,True,...,77.791,28.000,102.000,195.000,205.000,0.0,147.0,56.155,64.224,25.000
94300,3000747,2017-08-15,57.842,0,15,33,8,2017,False,True,...,64.859,77.791,28.000,102.000,195.000,205.0,0.0,147.000,56.155,64.224
94301,3000751,2017-08-15,59.619,0,15,33,8,2017,False,True,...,41.000,64.859,77.791,28.000,102.000,195.0,205.0,0.000,147.000,56.155
94302,3000752,2017-08-15,94.000,0,15,33,8,2017,False,True,...,2.000,41.000,64.859,77.791,28.000,102.0,195.0,205.000,0.000,147.000


### Dealing with Categorical Columns

Now that we have the windowed dataframe, we need to take care of the categorical columns. Some of the columns are binary, we can replace then with 0 or 1. The *type* column, however, needs to be converted using one-hot encoding.

We also need to drop some columns which are not needed for the model.

In [82]:
df.dtypes.head(15)

id               int64
date            object
sales          float64
onpromotion      int64
day              int64
week             int64
month            int64
year             int64
isweekend         bool
wage_day          bool
type            object
oil_price      float64
holiday           bool
sales_1        float64
sales_2        float64
dtype: object

In [133]:
id = df["id"]
df = df.drop(["id", "date"], axis=1)

We need to drop both the `date` and the `id` columns. Also, as we are going to need `id` for later, we'll be saving this to a separate dataframe.

`isweekend` is a binary column. We can replace it with 0 or 1. The same can be done with the `wage_day` and `holiday` columns.

In [134]:
df["type"].value_counts()

C    94248
Name: type, dtype: int64

Thr `type` columns contains four unique values. We need to convert them to one-hot encoding.

In [135]:
binary_cols = ["isweekend", "wage_day", "holiday"]
one_hot_cols = ["type"]

There are also some ordinal columns, like *day*, *month*, *year* and *week*. We will leave them for now.

Apart from these, we need to scale some of the numerical columns. Theses columns *sales* and their windowed counterparts. We will be leaving them for now.

In [136]:
one_hot_encoder = OneHotEncoder(sparse=False, handle_unknown="ignore")

In [137]:
column_transformer = ColumnTransformer(
    [
        ("boolian_cols", one_hot_encoder, binary_cols),
        ("categorical_cols", one_hot_encoder, one_hot_cols),
    ],
    remainder="passthrough",
)

### Train Test Split

We have our column transformer ready to be used. Let's do train test split.

In [141]:
y = df["sales"]
X = df.drop(["sales"], axis=1)

X_train = X[:int(len(df) *0.8)]
y_train = y[:int(len(df) *0.8)]
X_test = X[int(len(df) *0.8):]
y_test = y[int(len(df) *0.8):]

In [142]:
len(X_train), len(y_train), len(X_test), len(y_test)

(75398, 75398, 18850, 18850)

In [143]:
X_train = column_transformer.fit_transform(X_train)
X_test = column_transformer.transform(X_test)

Now everything is ready to go!

# Training a Simple Model

In [144]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [148]:
mse = mean_squared_error(y_test, lr.predict(X_test))
mse

7444.154026052979

In [149]:
mae = mean_absolute_error(y_test, lr.predict(X_test))
mae

39.44609619478573

In [154]:
df["sales"].describe()

count    94248.000000
mean       100.548931
std        106.434228
min          0.000000
25%         28.000000
50%         70.496000
75%        144.000000
max       3768.671000
Name: sales, dtype: float64

So, the linear model is performing good enough. The MAE is less than 30% of the mean value.

In [155]:
y_pred = lr.predict(X_test)
y_pred

array([ 33.8894043 ,  19.52929688, 274.72460938, ...,  61.28112793,
        99.19506836,  -0.44824219])

In [158]:
y_test.values

array([ 11.   ,   2.   , 264.   , ...,  59.619,  94.   ,   3.   ])

It seems our model is giving some negative values. To remove thi problem, we can train our model on the log of the sales. Let's do this next.

### Training the model on log of price

In [169]:
prices_and_oil = list(df.columns[df.dtypes == np.float64])
prices_and_oil.remove("oil_price")
prices_and_oil.remove("sales")
prices_and_oil

['sales_1',
 'sales_2',
 'sales_3',
 'sales_4',
 'sales_5',
 'sales_6',
 'sales_7',
 'sales_8',
 'sales_9',
 'sales_10',
 'sales_11',
 'sales_12',
 'sales_13',
 'sales_14',
 'sales_15',
 'sales_16',
 'sales_17',
 'sales_18',
 'sales_19',
 'sales_20',
 'sales_21',
 'sales_22',
 'sales_23',
 'sales_24',
 'sales_25',
 'sales_26',
 'sales_27',
 'sales_28',
 'sales_29',
 'sales_30',
 'sales_31',
 'sales_32',
 'sales_33',
 'sales_34',
 'sales_35',
 'sales_36',
 'sales_37',
 'sales_38',
 'sales_39',
 'sales_40',
 'sales_41',
 'sales_42',
 'sales_43',
 'sales_44',
 'sales_45',
 'sales_46',
 'sales_47',
 'sales_48',
 'sales_49',
 'sales_50',
 'sales_51',
 'sales_52',
 'sales_53',
 'sales_54',
 'sales_55',
 'sales_56']

In [170]:
log_cols = prices_and_oil
log_transformer = FunctionTransformer(np.log1p, validate=False)

column_transformer = ColumnTransformer(
    [
        ("boolian_cols", one_hot_encoder, binary_cols),
        ("categorical_cols", one_hot_encoder, one_hot_cols),
        ("log_cols", log_transformer, log_cols),
    ],
    remainder="passthrough",
)

In [189]:
y = df["sales"]
X = df.drop(["sales"], axis=1)
y = log_transformer.fit_transform(y)
X_train = X[:int(len(df) *0.8)]
y_train = y[:int(len(df) *0.8)]
X_test = X[int(len(df) *0.8):]
y_test = y[int(len(df) *0.8):]

In [190]:
X_train = column_transformer.fit_transform(X_train)
X_test = column_transformer.transform(X_test)

In [173]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [191]:
preds = lr.predict(X_test)
preds = np.exp(preds)-1
y_test = np.exp(y_test)-1

mse = mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)

mse, mae

(7185.0031073736845, 37.31601408833049)

In [192]:
preds

array([ 17.6557627 ,   0.82131685, 252.0011059 , ...,  60.34351192,
        74.78901328,   0.95246775])

In [194]:
y_test.values

array([ 11.   ,   2.   , 264.   , ...,  59.619,  94.   ,   3.   ])

This is giving better result. Plus, we are not getting any negative values.

## Training A Simple Model on Unwindowed Data

In [203]:
df = df.drop(log_cols, axis=1)

In [255]:
y = df["sales"]
X = df.drop(["sales"], axis=1)
y = log_transformer.fit_transform(y)
X_train = X[:int(len(df) *0.8)]
y_train = y[:int(len(df) *0.8)]
X_test = X[int(len(df) *0.8):]
y_test = y[int(len(df) *0.8):]

In [256]:
column_transformer = ColumnTransformer(
    [
        ("boolian_cols", one_hot_encoder, binary_cols),
        ("categorical_cols", one_hot_encoder, one_hot_cols),
        # ("log_cols", log_transformer, log_cols),
    ],
    remainder="passthrough",
)

In [257]:
X_train = column_transformer.fit_transform(X_train)
X_test = column_transformer.transform(X_test)

In [258]:
lr.fit(X_train, y_train)

LinearRegression()

In [259]:
preds = lr.predict(X_test)
preds = np.exp(preds)-1
y_test = np.exp(y_test)-1

mae = mean_absolute_error(y_test, preds)
mse = mean_squared_error(y_test, preds)
msle = mean_squared_log_error(y_test, preds)
mae, mse, msle, np.sqrt(msle)

(74.56723625893505, 15783.97110771318, 1.9283836919285453, 1.388662555097006)

In [260]:
preds = lr.predict(X_train)
preds = np.exp(preds)-1
y_train = np.exp(y_train)-1

mae = mean_absolute_error(y_train, preds)
mse = mean_squared_error(y_train, preds)
msle = mean_squared_log_error(y_train, preds)
mae, mse, msle, np.sqrt(msle)

(71.10414327709802, 12583.209317801384, 2.1862944562158693, 1.4786123414255237)

In [261]:
y_test = np.log1p(y_test)
y_train = np.log1p(y_train)

It seems that the unwindowed data is giving better results. 

# Creating Some Functions

There are quite a few steps involved to train a model for the given dataset. Let's make some functions to make it easier.

## Transforming the Data

To transform the data for the model, we'll merge the train and test dataframes. This is done because otherwise the number of columns is coming out to be different.

In [433]:
def merge_dataframes(file, log_sale=True):
    train_data = pd.read_csv(os.path.join("data", "train", file)).set_index("id")
    test_data = pd.read_csv(os.path.join("data", "test", file)).set_index("id")

    train_indices = train_data.index
    test_indices = test_data.index
    y_train = train_data.sales.values

    merged_df = pd.concat([train_data, test_data], axis=0)
    merged_df = merged_df.drop(["date", "sales"], axis=1)
    merged_indices = merged_df.index

    one_hot_cols = merged_df.columns[(merged_df.dtypes == "object")|(merged_df.dtypes == "bool")]
    one_hot_encoder = OneHotEncoder(sparse=False, handle_unknown="error", drop="first")
    column_transformer = ColumnTransformer(
    [
        ("categorical_cols", one_hot_encoder, one_hot_cols),
    ],
    remainder="passthrough",
    )
    if log_sale:
        y_train = np.log1p(y_train)

    merged_df = column_transformer.fit_transform(merged_df)
    merged_df = pd.DataFrame(merged_df, index=merged_indices)
    train_data = merged_df.loc[train_indices]
    test_data = merged_df.loc[test_indices]
    return (train_data, test_data, y_train), test_indices

In [434]:
(train_data, test_data, y_train), test_id= merge_dataframes("A_FOOD.csv")

In [435]:
train_data

,0,1,2,3,4,5,6,7,8
id,,,,,,,,,
236,0.0,1.0,1.0,0.0,1.0,1.0,1.0,2013.0,93.055
240,0.0,1.0,1.0,0.0,1.0,1.0,1.0,2013.0,93.055
241,0.0,1.0,1.0,0.0,1.0,1.0,1.0,2013.0,93.055
242,0.0,1.0,1.0,0.0,1.0,1.0,1.0,2013.0,93.055
255,0.0,1.0,1.0,0.0,1.0,1.0,1.0,2013.0,93.055
...,...,...,...,...,...,...,...,...,...
3000734,0.0,1.0,1.0,0.0,15.0,33.0,8.0,2017.0,47.570
3000747,0.0,1.0,1.0,0.0,15.0,33.0,8.0,2017.0,47.570
3000751,0.0,1.0,1.0,0.0,15.0,33.0,8.0,2017.0,47.570


In [436]:
test_data

,0,1,2,3,4,5,6,7,8
id,,,,,,,,,
3001124,0.0,0.0,0.0,15.0,16.0,33.0,8.0,2017.0,46.80
3001128,0.0,0.0,0.0,18.0,16.0,33.0,8.0,2017.0,46.80
3001129,0.0,0.0,0.0,1.0,16.0,33.0,8.0,2017.0,46.80
3001130,0.0,0.0,0.0,1.0,16.0,33.0,8.0,2017.0,46.80
3001143,0.0,0.0,0.0,0.0,16.0,33.0,8.0,2017.0,46.80
...,...,...,...,...,...,...,...,...,...
3029246,0.0,0.0,0.0,1.0,31.0,35.0,8.0,2017.0,47.26
3029259,0.0,0.0,0.0,9.0,31.0,35.0,8.0,2017.0,47.26
3029263,0.0,0.0,0.0,0.0,31.0,35.0,8.0,2017.0,47.26


In [437]:
test_id

Int64Index([3001124, 3001128, 3001129, 3001130, 3001143, 3001147, 3001148,
            3001151, 3001652, 3001656,
            ...
            3028769, 3028772, 3029240, 3029244, 3029245, 3029246, 3029259,
            3029263, 3029264, 3029267],
           dtype='int64', name='id', length=896)

In [438]:
y_train

array([0.        , 0.        , 0.        , ..., 4.10460838, 4.55387689,
       1.38629436])

## Training Model

### Creating Train Test Split

In [439]:
def create_train_test(X, y, ratio=0.8):
    X = X.values
    X_train = X[:int(len(X) * ratio)]
    y_train = y[:int(len(y) * ratio)]
    X_test = X[int(len(X) * ratio):]
    y_test = y[int(len(y) * ratio):]
    return X_train, X_test, y_train, y_test

In [440]:
X_train, X_test, y_train, y_test = create_train_test(train_data, y_train)

In [441]:
len(X_train), len(X_test)

(75443, 18861)

### Evaluating the Model

In [442]:
def evaluate(model, on="test", log_sale=True, return_preds = True):
    if on == "test":
        X = X_test
        y = y_test
    elif on == "train":
        X = X_train
        y = y_train
    print("Predicting")
    preds = model.predict(X)
    if log_sale:
        print("Exponentiating")
        preds = np.exp(preds)-1
        y = np.exp(y)-1
    print("Calculating Metrics")
    mse = mean_squared_error(y, preds)
    mae = mean_absolute_error(y, preds)
    msle = mean_squared_log_error(y, preds)
    rmse = np.sqrt(msle)
    print("MSE:", mse)
    print("MAE:", mae)
    print("MSLE:", msle)
    print("RMSE:", rmse)
    
    metrics = {"mse": mse, "mae": mae, "msle": msle, "rmse": rmse}
    if return_preds:
        return metrics, preds
    else:
        return metrics

In [443]:
lr = LinearRegression()
lr.fit(X_train, y_train)
metrics, preds = evaluate(lr, on="test")

Predicting
Exponentiating
Calculating Metrics
MSE: 15794.13822505988
MAE: 74.57480769428126
MSLE: 1.932362905052333
RMSE: 1.3900945669458367


### Creating The Prediction DataFrame

In [447]:
def create_pred_dataframe(X, model, id, log_sale=True):
    preds = model.predict(X)
    if log_sale:
        preds = np.exp(preds)-1
    df = pd.DataFrame(preds, columns=["sales"])
    df = pd.DataFrame(id, columns=["id"])
    df["sales"] = preds
    df.reset_index(drop=True, inplace=True)
    return df

In [449]:
create_pred_dataframe(test_data, lr, test_id)

,id,sales
0,3001124,112.376955
1,3001128,130.258794
2,3001129,56.241087
3,3001130,56.241087
4,3001143,53.513826
...,...,...
891,3029246,55.153741
892,3029259,81.982852
893,3029263,52.478287
894,3029264,52.478287


Using these functions, we can easily create the model and evaluate it as well as create the dataframe for the predictions. This is what we'll be doing in the next notebook.